# Step 3 — Filtering the short leg for borrow cost (Section 4)

Build a point-in-time hard-to-borrow (HTB) proxy on the Step 2 eligible pool and
map it to the Section 6 borrow schedule. Logic lives in `step3_borrow_cost.py`;
`python step3_borrow_cost.py` reproduces every output.

| Tier | Meaning | Rate |
|------|---------|------|
| A | General Collateral | 40 bps p.a. |
| B | mid-tier special | 200 bps p.a. |
| C | deep special | 800 bps p.a. |

Daily charge = annual rate / 252. *Full justification → report.*

In [ ]:
import pandas as pd
import step3_borrow_cost as s3

elig, si = s3.load_inputs()
base = elig[elig["is_trade_eligible"]]
cov = base.merge(si[["date","instrument_id","dsi"]], on=["date","instrument_id"], how="left")["dsi"].notna().mean()
print(f"Eligible stock-days: {len(base):,} | short-interest coverage: {cov*100:.2f}%")

## Q1 — HTB proxy

Inputs: `dsi` (SI/shares out), `dtcn` (days-to-cover), `ddtcn` (borrow stress).
Thresholds read off the eligible-pool distribution below — Tier B at `dsi>=0.10`
or `dtcn>=10`, Tier C at `dsi>=0.20`/`dtcn>=20` or near-deep + acute stress.

In [2]:
d = base.merge(si[["date","instrument_id","dsi","dtcn","ddtcn"]],
               on=["date","instrument_id"], how="left").dropna(subset=["dsi","dtcn"])
q = [0.5, 0.75, 0.90, 0.95, 0.99]
dist = pd.DataFrame({"dsi": d["dsi"].quantile(q), "dtcn": d["dtcn"].quantile(q)}).round(3)
dist.index = [f"p{int(x*100)}" for x in q]
print(dist.to_string())

       dsi    dtcn
p50  0.026   3.543
p75  0.050   5.898
p90  0.089   9.544
p95  0.124  12.595
p99  0.204  21.519


In [3]:
tiers = s3.build_borrow_tiers(elig, si)
print(tiers["borrow_tier"].value_counts(normalize=True).mul(100).round(2).sort_index().to_string())
print(f"HTB share (B+C): {(tiers['borrow_tier']!='A').mean()*100:.2f}%")
tiers.head()

borrow_tier
A    87.02
B     9.53
C     3.45
HTB share (B+C): 12.98%


,date,year,ticker,instrument_id,dsi,dtcn,ddtcn,hard_to_borrow_flag,borrow_stress_flag,no_short_interest,borrow_tier,borrow_rate_annual_bps,borrow_cost_daily_bps
0,2015-03-03,2015,HLT,1,0.004025,1.134529,-0.362399,False,False,False,A,40.0,0.15873
1,2015-03-04,2015,HLT,1,0.004025,1.134529,-0.362399,False,False,False,A,40.0,0.15873
2,2015-03-05,2015,HLT,1,0.004025,1.134529,-0.362399,False,False,False,A,40.0,0.15873
3,2015-03-06,2015,HLT,1,0.004025,1.134529,-0.362399,False,False,False,A,40.0,0.15873
4,2015-03-09,2015,HLT,1,0.004166,0.889116,-2.261740,False,False,False,A,40.0,0.15873


## Q3 — Treatment: tiered cost (not hard exclusion)

`hard_to_borrow_flag` retained for the Section 7.4 exclusion robustness check.
~13% of eligible stock-days are HTB; yearly breakdown below.

In [4]:
yearly = s3.tier_yearly_summary(tiers)
print(yearly.to_string(index=False))

 year      A     B     C  total  pct_A  pct_B  pct_C  htb_share_pct
 2010 108484 16511  6285 131280  82.64  12.58   4.79          17.36
 2011 141994 19974  7320 169288  83.88  11.80   4.32          16.12
 2012 144344 18405  7474 170223  84.80  10.81   4.39          15.20
 2013 154836 19240 11896 185972  83.26  10.35   6.40          16.74
 2014 173867 24118 12128 210113  82.75  11.48   5.77          17.25
 2015 179249 25249 10593 215091  83.34  11.74   4.92          16.66
 2016 179617 29382 11093 220092  81.61  13.35   5.04          18.39
 2017 183863 29809 11060 224732  81.81  13.26   4.92          18.19
 2018 197465 26736  7779 231980  85.12  11.53   3.35          14.88
 2019 200563 24623  8088 233274  85.98  10.56   3.47          14.02
 2020 207787 18159  5765 231711  89.68   7.84   2.49          10.32
 2021 219495 14275  3463 237233  92.52   6.02   1.46           7.48
 2022 223398 10692  2324 236414  94.49   4.52   0.98           5.51
 2023 220214 12990  2078 235282  93.60   5.52   

## Q2 — External validation

Proxy surfaces documented HTB / squeeze names without hard-coding any ticker
(GME flagged from 2013, pre-squeeze).

In [6]:
print(s3.top_htb_names(tiers, n=15).to_string(index=False))

ticker  tier_c_days  mean_dsi  max_dsi  mean_dtcn first_date  last_date
  FFIN         2500    0.1124   0.1787      37.87 2010-03-03 2024-11-20
  ENOV         1557    0.2832   0.5524      33.04 2012-01-11 2022-02-04
   GME         1276    0.2861   0.4792      15.12 2013-04-02 2024-05-03
  GATX         1186    0.1996   0.3302      28.16 2010-03-11 2020-12-31
  PRAA         1110    0.1905   0.2811      27.65 2011-01-03 2017-12-29
   WBD         1100    0.2047   0.3630      11.83 2010-08-05 2022-05-05
     X         1098    0.2712   0.4038       4.66 2010-12-06 2021-11-18
    RH         1007    0.2725   0.3720       8.40 2014-08-21 2020-12-31
  MYGN          959    0.3582   0.5469      27.40 2013-10-04 2019-12-16
   BFH          904    0.2434   0.3664      22.31 2010-03-03 2013-12-19
   BKE          873    0.1454   0.2062      22.61 2010-03-03 2016-12-30
   GGG          852    0.0476   0.0844      28.48 2010-03-08 2017-12-20
   RLI          789    0.0667   0.0871      21.56 2010-04-05 201

## Q4 — Gross vs net Sharpe → Step 5

Needs realised returns. Step 3 hands off per (stock, date):
`borrow_tier`, `borrow_rate_annual_bps`, `borrow_cost_daily_bps` (charged on gross
short notional). Save outputs:

In [7]:
s3.main()

Step 3 complete. 3,171,067 eligible stock-days tiered.
Full-sample tier shares (%):
borrow_tier
A    87.02
B     9.53
C     3.45
Hard-to-borrow share (Tier B+C): 12.98%
No-short-interest stock-days defaulted to Tier A: 0.58%

Outputs written to /Users/xiuhaoyu/Desktop/machine learning/handoff_stage3_from_stage1_2/step3_outputs
ticker  tier_c_days  mean_dsi  max_dsi  mean_dtcn first_date  last_date
  FFIN         2500    0.1124   0.1787      37.87 2010-03-03 2024-11-20
  ENOV         1557    0.2832   0.5524      33.04 2012-01-11 2022-02-04
   GME         1276    0.2861   0.4792      15.12 2013-04-02 2024-05-03
  GATX         1186    0.1996   0.3302      28.16 2010-03-11 2020-12-31
  PRAA         1110    0.1905   0.2811      27.65 2011-01-03 2017-12-29
   WBD         1100    0.2047   0.3630      11.83 2010-08-05 2022-05-05
     X         1098    0.2712   0.4038       4.66 2010-12-06 2021-11-18
    RH         1007    0.2725   0.3720       8.40 2014-08-21 2020-12-31
  MYGN          959    